In [ ]:
# %%
# 6) Example task showing when the agent should hand off or ask for approval

GOAL = "Log in, then transfer $25 from my account to 999999."

print("Goal:", GOAL)
print("- Missing values: request_value / request_missing_values")
print("- Uncertain step: ask_human")
print("- Risky click/submit: approval before action")


In [ ]:
# %%
# 5) Build the deep agent

SYSTEM_PROMPT = """
You are a browser agent for a banking demo site.

Rules:
1. Observe the page before acting.
2. Use only values the user gave you. Never invent one.
3. If a value is missing, ask the human for it.
4. If you are unsure, ask the human.
5. Any action that changes data must stop for approval.
6. Finish with a clear summary and the values you found.
"""

agent = create_deep_agent(
    model=MODEL,
    tools=BROWSER_TOOLS,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=MemorySaver(),
)

print("Agent built successfully.")


In [ ]:
# %%
# 4) Browser tools aligned with the discovery agent pattern


@tool(parse_docstring=True)
async def observe() -> str:
    """Look at the current page and return the page text."""
    return "Current page observed."


@tool(parse_docstring=True)
async def type_secret(ref: str, name: str) -> str:
    """Type a stored secret in a form without exposing the value to the model."""
    value = resolve_secret(name)
    return f"Typed secret '{name}' into {ref}. Value hidden from the model."


@tool(parse_docstring=True)
async def type_text(ref: str, text: str) -> str:
    """Type a value only if it was explicitly supplied by the user."""
    return f"Typed '{text}' into {ref}."


@tool(parse_docstring=True)
async def click(ref: str) -> str:
    """Click an element.

    If the action submits or changes data, approval must be granted before it continues.
    """
    if "submit" in ref.lower() or "transfer" in ref.lower() or "payment" in ref.lower():
        approved = await require_approval("click", f"Click on {ref}")
        if not approved:
            return f"DECLINED: {ref} was not approved."
    return f"Clicked {ref}."


@tool(parse_docstring=True)
async def request_value(ref: str, hint: str) -> str:
    """Ask the human for a missing field value."""
    return human_handoff(f"Please fill {ref} ({hint}) and then hand back to the agent.")


@tool(parse_docstring=True)
async def request_missing_values(hints: dict[str, str] = {}) -> str:
    """Ask for every empty field on the current form in one go."""
    labels = "; ".join(f"{k}: {v}" for k, v in hints.items()) or "the current form"
    return human_handoff(f"Please fill in the missing values for {labels} and then hand back to the agent.")


@tool(parse_docstring=True)
async def ask_human(question: str) -> str:
    """Use this when the agent is unsure which element to choose or what to do next."""
    return human_handoff(question)


@tool(parse_docstring=True)
async def finish(summary: str, values: dict[str, str]) -> str:
    """End the run and report the final result."""
    RESULT.clear()
    RESULT.update({"summary": summary, "values": values})
    return "Recorded. Stop now."


BROWSER_TOOLS = [observe, type_secret, type_text, click, request_value, request_missing_values, ask_human, finish]
print("Browser tools ready:", [t.name for t in BROWSER_TOOLS])


In [ ]:
# %%
# 3) Human-offhand and approval patterns from the MDs

RESULT = {}


def human_handoff(message: str) -> str:
    """A simple stand-in for the handoff flow used by the real browser agent."""
    return f"HUMAN HANDOFF: {message}"


async def require_approval(action_name: str, description: str) -> bool:
    """Gate risky actions behind explicit human approval."""
    print(f"Approval needed for: {action_name} -> {description}")
    answer = input("Approve this action? [y/N]: ").strip().lower()
    return answer in {"y", "yes"}


print("Handoff and approval flow is ready.")


In [ ]:
# %%
# 2) Browser helper and surface


async def open_browser():
    """Open the browser and land on the login page."""
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=False)
    context = await browser.new_context(viewport={"width": 1280, "height": 900})
    page = await context.new_page()
    await page.goto(f"{BASE}/index.htm")
    return pw, browser, context, page


class BrowserSurface:
    def __init__(self, page):
        self.page = page

    async def observe(self) -> str:
        return await self.page.inner_text("body")

    async def click(self, selector: str) -> None:
        await self.page.locator(selector).click(timeout=5000, force=True)

    async def fill(self, selector: str, value: str) -> None:
        await self.page.locator(selector).fill(value, timeout=5000, force=True)

    async def page_text(self) -> str:
        return (await self.page.inner_text("body"))[:4000]


print("Browser helper ready.")


In [ ]:
# %%
# 1) Setup and environment
import os
from urllib.parse import urlparse

from dotenv import load_dotenv
from deepagents import create_deep_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from playwright.async_api import async_playwright

load_dotenv(override=True)

MODEL = os.getenv("MODEL", "anthropic:claude-sonnet-5")
BASE = "https://parabank.parasoft.com/parabank"
ALLOWED_HOSTS = {"parabank.parasoft.com"}
SECRETS = {"username": "PARABANK_USERNAME", "password": "PARABANK_PASSWORD"}


def resolve_secret(name: str) -> str:
    """Read a secret from the environment without exposing it to the model."""
    key = SECRETS.get(name)
    if not key:
        raise KeyError(f"Unknown secret: {name!r}")
    value = os.getenv(key, "")
    if not value:
        raise RuntimeError(f"Environment variable {key!r} is empty")
    return value


def host_allowed(url: str) -> bool:
    if url == "about:blank":
        return True
    return urlparse(url).hostname in ALLOWED_HOSTS


print("model:", MODEL)
print("base:", BASE)


# Discovery agent pattern from the repo MDs

This notebook is a compact implementation of the patterns described in the project docs and the existing browser-agent notebook:

- build an agent with `create_deep_agent(...)`
- use a human handoff when the agent cannot complete a step by itself
- ask for approval before a risky data-changing action
- request missing values without guessing
- stop cleanly with a final summary

This is intentionally simple and readable, but it follows the same logic as the project guidance.


In [ ]:
# %%
# Build the deep agent the same way as the project notebook pattern.

SYSTEM_PROMPT = """
You are a browser agent for a banking demo site.

Rules:
1. Observe the page before acting.
2. Use only values the user gave you. Never invent one.
3. If a value is missing, ask the human for it and continue.
4. If you are unsure which element to choose, ask the human.
5. If an action changes data, stop for approval.
6. Finish by reporting summary and values.
"""

agent = create_deep_agent(
    model=MODEL,
    tools=BROWSER_TOOLS,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=MemorySaver(),
)

print("Agent built successfully.")


In [ ]:
# %%
# Browser tools: intentionally small, but aligned with the project agent patterns.


class BrowserSurface:
    def __init__(self, page):
        self.page = page

    async def observe(self) -> str:
        return await self.page.inner_text("body")

    async def click(self, selector: str) -> None:
        await self.page.locator(selector).click(timeout=5000, force=True)

    async def fill(self, selector: str, value: str) -> None:
        await self.page.locator(selector).fill(value, timeout=5000, force=True)

    async def page_text(self) -> str:
        return (await self.page.inner_text("body"))[:4000]


@tool(parse_docstring=True)
async def observe() -> str:
    """Look at the current page and return the visible text."""
    return "Current page observed."


@tool(parse_docstring=True)
async def type_secret(ref: str, name: str) -> str:
    """Type a stored secret into a field without revealing the value."""
    return f"Typed secret '{name}' into {ref}."


@tool(parse_docstring=True)
async def type_text(ref: str, text: str) -> str:
    """Type a value the user explicitly gave us."""
    return f"Typed '{text}' into {ref}."


@tool(parse_docstring=True)
async def click(ref: str) -> str:
    """Click an element on the page.

    If the action changes data or submits a form, this should require approval first.
    """
    if "submit" in ref.lower() or "transfer" in ref.lower() or "payment" in ref.lower():
        allow = await require_approval("click", f"Click on {ref}")
        if not allow:
            return f"DECLINED: {ref} was not approved."
    return f"Clicked {ref}."


@tool(parse_docstring=True)
async def request_value(ref: str, hint: str) -> str:
    """Ask the human to fill one missing field."""
    return human_handoff(f"Please fill {ref} ({hint}) and then hand back to the agent.")


@tool(parse_docstring=True)
async def request_missing_values(hints: dict[str, str] = {}) -> str:
    """Ask the human for every missing value on the current form in one pass."""
    labels = "; ".join(f"{k}: {v}" for k, v in hints.items()) or "the current form"
    return human_handoff(f"Please fill in the missing values for {labels} and then hand back to the agent.")


@tool(parse_docstring=True)
async def ask_human(question: str) -> str:
    """Use this when the agent is uncertain or needs a decision from a human."""
    return human_handoff(question)


@tool(parse_docstring=True)
async def finish(summary: str, values: dict[str, str]) -> str:
    """Report the final result and stop."""
    RESULT.clear()
    RESULT.update({"summary": summary, "values": values})
    return "Recorded. Stop now."


BROWSER_TOOLS = [observe, type_secret, type_text, click, request_value, request_missing_values, ask_human, finish]
print("Browser tools ready:", [t.name for t in BROWSER_TOOLS])


In [ ]:
# %%
# The key decision-making patterns used in the MD design notes.

RESULT = {}


def human_handoff(message: str) -> str:
    """Simple stand-in for the browser handoff used by the agent."""
    return f"HUMAN HANDOFF: {message}"


async def require_approval(action_name: str, description: str) -> bool:
    """Model a risky action gate before a state-changing click is allowed."""
    print(f"Approval needed for: {action_name} -> {description}")
    answer = input("Approve this action? [y/N]: ").strip().lower()
    return answer in {"y", "yes"}


print("Human handoff and approval gates are active.")


In [ ]:
# %%
# Browser setup and a minimal surface to mirror the real notebook behavior.

from urllib.parse import urlparse


async def open_browser():
    """Open a visible browser and land on the login page."""
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=False)
    context = await browser.new_context(viewport={"width": 1280, "height": 900})
    page = await context.new_page()
    await page.goto(f"{BASE}/index.htm")
    return pw, browser, context, page


def host_allowed(url: str) -> bool:
    if url == "about:blank":
        return True
    return urlparse(url).hostname in ALLOWED_HOSTS


print("Browser helper ready for the selected site.")


In [ ]:
# %%
# Setup: browser + secrets config from the project notebook pattern.
import os
from dotenv import load_dotenv
from deepagents import create_deep_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from playwright.async_api import async_playwright

load_dotenv(override=True)

MODEL = os.getenv("MODEL", "anthropic:claude-sonnet-5")
BASE = "https://parabank.parasoft.com/parabank"
ALLOWED_HOSTS = {"parabank.parasoft.com"}
SECRETS = {"username": "PARABANK_USERNAME", "password": "PARABANK_PASSWORD"}


def resolve_secret(name: str) -> str:
    """Read a secret from the environment without exposing it to the model."""
    key = SECRETS.get(name)
    if not key:
        raise KeyError(f"Unknown secret: {name!r}")
    value = os.getenv(key, "")
    if not value:
        raise RuntimeError(f"Environment variable {key!r} is empty")
    return value


print("model:", MODEL)
print("base:", BASE)


# Discovery Agent Pattern from the project MDs

This notebook implements the practical patterns described in the project docs and the working browser-agent notebook:

- how to build an agent with `create_deep_agent(...)`
- when to give a human offhand / handoff
- when approval is required before a risky action
- how to use `request_value`, `request_missing_values`, and `ask_human`
- how to finish with a clear summary and recorded values

The goal is to keep the workflow simple and faithful to the repo's design decisions.


## Summary

This notebook applies the repo’s decision patterns directly:

- use a human handoff when the agent lacks a value or is uncertain
- treat submit/payment/transfer actions as approval-gated
- build the agent once, then let it follow a narrow system prompt
- finish with a recorded result instead of continuing blindly

The design is intentionally compact so it stays easy to adapt to the real project flow.
